<a href="https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/02_convolutional_neural_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning for Maritime Vision Applications
## Session 0 · Part 3 — Convolutional Neural Networks
*Torben Globisch · VACOT*

---

| Notebook | Topic |
|---|---|
| 00 | Artificial Neural Networks: neurons, layers, learning |
| 01 | Computer Vision & an MLP on MNIST |
| **→ 02** | **Convolutional Neural Networks** |
| 03 | Vision Foundation Models: the CNN era |
| 04 | Vision Transformers |
| 05 | Self-supervised pretraining & transfer |

**By the end of this notebook you can**
1. compute a convolution by hand and explain what a kernel detects,
2. predict the output size of a conv / pooling layer from kernel size, stride and padding,
3. build and train a CNN on MNIST (**Exercise 3**) and compare it with the MLP from notebook 01,
4. explain *why* CNNs beat MLPs on images: parameter sharing, local connectivity, translation equivariance.

> 💡 Use a GPU runtime: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# @title ⚙️ Setup: imports, colours & helpers — run me first { display-mode: "form" }
import math, io, time, urllib.request
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrowPatch
from IPython.display import HTML, display
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

# Colour palette — the same colours as the lecture slides
NAVY, BLUE, LAVENDER, ORANGE, GREEN, RED, GREY, LIGHT = (
    "#2D3A8C", "#5B67B8", "#A3A9DB", "#E8702A", "#3E9E64", "#C0392B", "#8A8FA8", "#EEF0F7")
plt.rcParams.update({
    "figure.dpi": 100, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 11, "axes.titlecolor": NAVY,
    "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, GREEN, BLUE, RED, GREY]),
    "animation.html": "jshtml", "animation.embed_limit": 80,
})

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)
print(f"PyTorch {torch.__version__} | device: {device}")
if device.type == "cpu":
    print("⚠️  No GPU found. In Colab: Runtime → Change runtime type → T4 GPU")

def play(anim):
    """Show a matplotlib animation as an HTML player (play / pause / step frame by frame)."""
    plt.close(anim._fig)
    return HTML(anim.to_jshtml())

# Maritime example photos from the COCO validation set (Flickr, Creative Commons licences).
MARITIME = {"trawler": 330369, "sailboat": 286422, "lifeboat": 400803, "marina": 442822,
            "fishing_boat": 451090, "yacht": 489014, "ferry": 543300, "harbour": 532575}

def load_image(name, size=None):
    """Load one of the MARITIME photos (or any image URL) as an RGB PIL image."""
    url = f"http://images.cocodataset.org/val2017/{MARITIME[name]:012d}.jpg" if name in MARITIME else name
    with urllib.request.urlopen(url, timeout=30) as r:
        img = Image.open(io.BytesIO(r.read())).convert("RGB")
    if size is not None:  # centre-crop to a square, then resize
        s = min(img.size); l, t = (img.width - s) // 2, (img.height - s) // 2
        img = img.crop((l, t, l + s, t + s)).resize((size, size), Image.BICUBIC)
    return img

---
# 1 · The convolution operation

A **Convolutional Neural Network (CNN)** is designed for grid-structured data (images, spectrograms, …).

**Core idea:** instead of learning a separate weight for every pixel, learn **small filters (kernels)** that **slide**
across the image and detect local patterns. At every position we compute a dot product between the kernel and the image patch below it:

$$ (I * K)_{i,j} = \sum_{m=0}^{k-1}\sum_{n=0}^{k-1} I_{i+m,\; j+n}\cdot K_{m,n} $$

*(Strictly speaking this is a cross-correlation; deep-learning libraries call it convolution anyway.)*

## 1.1 · Walkthrough: detecting an edge with the Sobel kernel
Our input is an 8 × 8 greyscale image: dark on the left (pixel value 50), light on the right (200), with a vertical
edge between columns 3 and 4. We slide the 3 × 3 **Sobel-X** kernel over it:

$$ K = \begin{pmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{pmatrix} $$

Output size without padding: $(8-3+1) \times (8-3+1) = 6\times6$. Press ▶ or step through the positions one by one.

In [ ]:
I = np.full((8, 8), 50.0); I[:, 4:] = 200.0
K = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
out = np.zeros((6, 6))
for i in range(6):
    for j in range(6):
        out[i, j] = (I[i:i + 3, j:j + 3] * K).sum()
print(out.astype(int))

In [ ]:
# @title 🎞️ Animation: the kernel slides over the image { display-mode: "form" }
from matplotlib.colors import LinearSegmentedColormap
IN_CMAP = LinearSegmentedColormap.from_list("in", [BLUE, LIGHT])
OUT_CMAP = LinearSegmentedColormap.from_list("out", ["white", ORANGE])

def grid_axes(ax, n_rows, n_cols):
    ax.set_xticks(np.arange(-.5, n_cols), minor=True); ax.set_yticks(np.arange(-.5, n_rows), minor=True)
    ax.grid(which="minor", color="white", lw=1.5); ax.tick_params(which="both", length=0)
    ax.set_xticks(range(n_cols)); ax.set_yticks(range(n_rows))
    for s in ax.spines.values(): s.set_visible(False)

fig = plt.figure(figsize=(15, 5.2))
gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.05, 0.95])
ax_in, ax_txt, ax_out = fig.add_subplot(gs[0]), fig.add_subplot(gs[1]), fig.add_subplot(gs[2])
ax_in.imshow(I, cmap=IN_CMAP, vmin=0, vmax=255); grid_axes(ax_in, 8, 8); ax_in.set_title("Input image I (8×8)")
for (r, c), v in np.ndenumerate(I): ax_in.text(c, r, int(v), ha="center", va="center", fontsize=8, color="white" if v < 100 else NAVY)
ax_in.axvline(3.5, color=RED, ls="--", lw=1.5)
box = Rectangle((-0.5, -0.5), 3, 3, fill=False, ec=ORANGE, lw=3.5); ax_in.add_patch(box)
shown = np.full((6, 6), np.nan)
im_out = ax_out.imshow(shown, cmap=OUT_CMAP, vmin=0, vmax=600); grid_axes(ax_out, 6, 6); ax_out.set_title("Feature map I ∗ K (6×6)")
cell = Rectangle((-0.5, -0.5), 1, 1, fill=False, ec=NAVY, lw=3); ax_out.add_patch(cell)
out_txt = [[ax_out.text(c, r, "", ha="center", va="center", fontsize=9, color=NAVY) for c in range(6)] for r in range(6)]
ax_txt.axis("off"); calc = ax_txt.text(0.0, 0.95, "", family="monospace", fontsize=10.5, va="top", transform=ax_txt.transAxes)
verdict = ax_txt.text(0.0, 0.12, "", fontsize=13, fontweight="bold", transform=ax_txt.transAxes)
plt.tight_layout(); fig.subplots_adjust(top=0.9)

def update(t):
    i, j = divmod(t, 6)
    box.set_xy((j - 0.5, i - 0.5)); cell.set_xy((j - 0.5, i - 0.5))
    shown[i, j] = out[i, j]; im_out.set_data(shown); out_txt[i][j].set_text(int(out[i, j]))
    P = I[i:i + 3, j:j + 3]
    lines = [f"kernel at (row {i}, col {j})", f"covers columns {j}–{j + 2}", ""]
    for r in range(3):
        terms = " + ".join(f"({K[r, c]:+.0f})·{P[r, c]:.0f}" for c in range(3))
        lines.append(f"{terms} = {(P[r] * K[r]).sum():+5.0f}")
    lines += ["", f"sum = {out[i, j]:.0f}"]
    calc.set_text("\n".join(lines))
    if out[i, j] == 0:
        verdict.set_text("0 → uniform patch: left and right cancel"); verdict.set_color(GREY)
    else:
        verdict.set_text(f"{out[i, j]:.0f} → strong response: edge found!"); verdict.set_color(ORANGE)
    return [box, cell, im_out, calc, verdict]

play(animation.FuncAnimation(fig, update, frames=36, interval=500))

**Key insight.** The feature map has a bright stripe **exactly where the edge is**. The Sobel kernel computes a
*difference* between the right and left neighbours: zero on uniform regions, large where brightness changes.

## 1.2 · Convolution on a real image
Different kernels detect different things. Here are some classic hand-crafted ones applied with `F.conv2d`:

In [ ]:
gray = torch.tensor(np.asarray(load_image("trawler").convert("L")), dtype=torch.float32)[None, None] / 255   # (1, 1, H, W)

kernels = {
    "Sobel-X (vertical edges)": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
    "Sobel-Y (horizontal edges)": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
    "Laplacian (all edges)": [[0, 1, 0], [1, -4, 1], [0, 1, 0]],
    "Box blur": [[1 / 9] * 3] * 3,
    "Sharpen": [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
}
fig, axes = plt.subplots(2, 3, figsize=(15, 6.6))
axes[0, 0].imshow(gray[0, 0], cmap="gray"); axes[0, 0].set_title("input (greyscale)")
for ax, (name, k) in zip(axes.ravel()[1:], kernels.items()):
    kt = torch.tensor(k, dtype=torch.float32)[None, None]          # (out_ch, in_ch, 3, 3)
    res = F.conv2d(gray, kt)[0, 0]
    edge = "edge" in name.lower() or "laplacian" in name.lower()
    ax.imshow(res.abs() if edge else res.clamp(0, 1), cmap="magma" if edge else "gray")
    ax.set_title(name)
    ins = ax.inset_axes([0.0, 0.0, 0.22, 0.32]); ins.imshow(kt[0, 0], cmap="coolwarm"); ins.set_xticks([]); ins.set_yticks([])
    for (r, c), v in np.ndenumerate(kt[0, 0].numpy()): ins.text(c, r, f"{v:.2g}", ha="center", va="center", fontsize=6)
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout(); plt.show()

In a CNN we **do not hand-craft** these numbers. The kernel values are **parameters**, learned by backpropagation
exactly like the weights in notebook 00. (Notebook 03 shows a kernel *learning* to become a Sobel filter.)

---
# 2 · Stride and padding: controlling the output size

* **Stride $s$**: how many pixels the kernel jumps at each step. $s=1$ samples densely; $s=2$ roughly **halves** the output.
* **Padding $p$**: add zeros around the border. $p=0$ (*"valid"*) shrinks the output; $p=\lfloor k/2 \rfloor$ (*"same"*) keeps the size (for $s = 1$).

$$ \boxed{\;\text{output size} = \left\lfloor \frac{W - K + 2p}{s} \right\rfloor + 1\;} \qquad W=\text{input},\; K=\text{kernel},\; p=\text{padding},\; s=\text{stride}$$

In [ ]:
# @title 🎞️ Animation: stride and padding side by side { display-mode: "form" }
W_IN, KS = 7, 3
configs = [("stride 1, padding 0", 1, 0), ("stride 2, padding 0", 2, 0), ("stride 1, padding 1 ('same')", 1, 1)]

fig, axes = plt.subplots(2, 3, figsize=(14, 8.2), gridspec_kw=dict(height_ratios=[1.25, 1]))
state = []
for col, (name, s, p) in enumerate(configs):
    n_out = (W_IN - KS + 2 * p) // s + 1
    ax = axes[0, col]; Wp = W_IN + 2 * p
    for r in range(Wp):
        for c in range(Wp):
            pad = not (p <= r < p + W_IN and p <= c < p + W_IN)
            ax.add_patch(Rectangle((c, Wp - 1 - r), 1, 1, fc="white" if pad else LAVENDER, ec=GREY if pad else "white", ls="--" if pad else "-", lw=1))
            if pad: ax.text(c + .5, Wp - .5 - r, "0", ha="center", va="center", color=GREY, fontsize=8)
    ax.set_xlim(-.2, Wp + .2); ax.set_ylim(-.2, Wp + .2); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"{name}\n{W_IN}×{W_IN} input → {n_out}×{n_out} output", fontsize=11)
    kbox = Rectangle((0, Wp - KS), KS, KS, fill=True, fc=ORANGE, alpha=.35, ec=ORANGE, lw=3); ax.add_patch(kbox)
    axo = axes[1, col]; cells = {}
    for r in range(n_out):
        for c in range(n_out):
            cells[r, c] = Rectangle((c, n_out - 1 - r), 1, 1, fc="white", ec=GREEN, lw=1.2); axo.add_patch(cells[r, c])
    axo.set_xlim(-.2, max(n_out, 5) + .2); axo.set_ylim(-.2, max(n_out, 5) + .2); axo.set_aspect("equal"); axo.axis("off")
    axo.set_title(f"output {n_out}×{n_out}  =  ⌊({W_IN} − {KS} + 2·{p}) / {s}⌋ + 1", fontsize=10, color=GREEN)
    positions = [(r, c) for r in range(n_out) for c in range(n_out)]
    state.append((kbox, cells, positions, s, Wp))
plt.tight_layout()

def update(t):
    for kbox, cells, positions, s, Wp in state:
        k = min(t, len(positions) - 1); r, c = positions[k]
        kbox.set_xy((c * s, Wp - KS - r * s))
        for idx, (rr, cc) in enumerate(positions):
            cells[rr, cc].set_facecolor(GREEN if idx == k else ("#BFE3CC" if idx < k else "white"))
    return []

play(animation.FuncAnimation(fig, update, frames=max(len(st[2]) for st in state), interval=350))

### 🕹️ Output-size calculator
Check the formula against PyTorch's own `nn.Conv2d`:

In [ ]:
from ipywidgets import interact, IntSlider

def conv_output_size(W, K, s=1, p=0):
    return (W - K + 2 * p) // s + 1

def calculator(W=28, K=3, stride=1, padding=0):
    formula = conv_output_size(W, K, stride, padding)
    torch_out = nn.Conv2d(1, 1, K, stride=stride, padding=padding)(torch.zeros(1, 1, W, W)).shape[-1] if formula > 0 else "error"
    print(f"⌊({W} − {K} + 2·{padding}) / {stride}⌋ + 1 = {formula}     nn.Conv2d says: {torch_out}")

interact(calculator, W=IntSlider(28, 4, 64), K=IntSlider(3, 1, 11), stride=IntSlider(1, 1, 8), padding=IntSlider(0, 0, 5));

---
# 3 · Pooling: spatial down-sampling
**Max pooling** keeps the strongest activation in each window. A 2 × 2 window with stride 2 halves width and height.
It has **no parameters**; it only summarises.

In [ ]:
# @title 🎞️ Animation: 2×2 max pooling { display-mode: "form" }
fm = np.array([[1, 3, 2, 0], [5, 6, 1, 2], [0, 2, 9, 4], [3, 1, 4, 8]])
pooled = fm.reshape(2, 2, 2, 2).max(axis=(1, 3))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 3.8), gridspec_kw=dict(width_ratios=[1.6, 1]))
a1.imshow(fm, cmap=OUT_CMAP, vmin=0, vmax=10); grid_axes(a1, 4, 4); a1.set_title("feature map 4×4")
for (r, c), v in np.ndenumerate(fm): a1.text(c, r, v, ha="center", va="center", fontsize=14, color=NAVY)
win = Rectangle((-.5, -.5), 2, 2, fill=False, ec=GREEN, lw=4); a1.add_patch(win)
shown_p = np.full((2, 2), np.nan); im_p = a2.imshow(shown_p, cmap=OUT_CMAP, vmin=0, vmax=10); grid_axes(a2, 2, 2)
a2.set_title("max-pooled 2×2"); txt_p = [[a2.text(c, r, "", ha="center", va="center", fontsize=16, color=NAVY) for c in range(2)] for r in range(2)]
plt.tight_layout()
def update(t):
    t = min(t, 3); r, c = divmod(t, 2)
    win.set_xy((2 * c - .5, 2 * r - .5)); shown_p[r, c] = pooled[r, c]; im_p.set_data(shown_p); txt_p[r][c].set_text(pooled[r, c])
    return [win, im_p]
play(animation.FuncAnimation(fig, update, frames=5, interval=900))

---
# 4 · Translation equivariance → invariance
The *same* kernel is applied at *every* position. So if the object moves, the feature map moves **with it**
(**equivariance**). Take the maximum over the whole map (global max pooling) and the result **doesn't change at all**
(**invariance**).

In [ ]:
# @title 🎞️ Animation: a digit moves, its feature map moves with it { display-mode: "form" }
from torchvision import datasets
mnist_train = datasets.MNIST("data", train=True, download=True)
mnist_test = datasets.MNIST("data", train=False, download=True)
digit = mnist_train.data[7].float() / 255
sobel = torch.tensor(K, dtype=torch.float32)[None, None]

def feature_map(shift):
    canvas = torch.zeros(1, 1, 36, 48); canvas[0, 0, 4:32, shift:shift + 28] = digit
    return canvas[0, 0], F.relu(F.conv2d(canvas, sobel))[0, 0]

py, px = np.unravel_index(feature_map(0)[1].argmax().item(), (34, 46))   # a pixel on the digit's strongest edge at shift 0
frames_data = []
for shift in list(range(0, 21, 2)) + list(range(20, -1, -2)):
    canvas, fmap = feature_map(shift)
    frames_data.append((canvas, fmap, fmap.max().item(), fmap[py, px].item()))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), gridspec_kw=dict(width_ratios=[1.3, 1.3, .9]))
im_c = axes[0].imshow(frames_data[0][0], cmap="gray_r"); axes[0].set_title("input: the digit moves →"); axes[0].axis("off")
im_f = axes[1].imshow(frames_data[0][1], cmap="magma", vmin=0, vmax=max(f[2] for f in frames_data)); axes[1].axis("off")
axes[1].set_title("feature map (Sobel-X + ReLU) moves along\n= translation EQUIVARIANCE")
axes[1].plot(px, py, "s", ms=8, mfc="none", mec=GREEN, mew=2)
bars = axes[2].bar(["global max\n(invariant)", "value at the\ngreen pixel"], [0, 0], color=[NAVY, GREEN])
axes[2].set_ylim(0, max(f[2] for f in frames_data) * 1.1); axes[2].set_title("after pooling")
plt.tight_layout()
def update(t):
    c, f, mx, px = frames_data[t]
    im_c.set_data(c); im_f.set_data(f); bars[0].set_height(mx); bars[1].set_height(px)
    return [im_c, im_f, *bars]
play(animation.FuncAnimation(fig, update, frames=len(frames_data), interval=150))

---
# 5 · Building a CNN: dimension walkthrough
A typical CNN alternates **Conv → ReLU → Pool** blocks (*feature extraction*) and ends with dense layers (*classifier*).
Here is the network for Exercise 3. Let's trace the tensor shape through every layer.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3), nn.ReLU(),   # 1×28×28  → 16×26×26
            nn.MaxPool2d(2),                              # 16×26×26 → 16×13×13
            nn.Conv2d(16, 32, kernel_size=3), nn.ReLU(),  # 16×13×13 → 32×11×11
            nn.MaxPool2d(2),                              # 32×11×11 → 32×5×5
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                 # 32×5×5   → 800
            nn.Linear(32 * 5 * 5, 128), nn.ReLU(),        # 800      → 128
            nn.Linear(128, 10),                           # 128      → 10
        )

    def forward(self, x):
        if x.dim() == 3: x = x.unsqueeze(1)              # (B, 28, 28) → (B, 1, 28, 28)
        return self.classifier(self.features(x))

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

x = torch.zeros(1, 1, 28, 28)
def short(layer):
    if isinstance(layer, nn.Conv2d): return f"Conv2d({layer.in_channels}→{layer.out_channels}, k={layer.kernel_size[0]})"
    if isinstance(layer, nn.Linear): return f"Linear({layer.in_features}→{layer.out_features})"
    if isinstance(layer, nn.MaxPool2d): return f"MaxPool2d({layer.kernel_size})"
    return type(layer).__name__

print(f"{'layer':<26}{'output shape':<16}{'params':>8}")
print(f"{'input':<26}{str(tuple(x.shape[1:])):<16}")
for layer in list(SimpleCNN().features) + list(SimpleCNN().classifier):
    x = layer(x)
    print(f"{short(layer):<26}{str(tuple(x.shape[1:])):<16}{count_params(layer):>8,}")
print(f"\ntotal parameters: {count_params(SimpleCNN()):,}")

In [ ]:
# @title 🎨 Figure: the shape of the data through the network { display-mode: "form" }
def draw_volume(ax, x, c, h, label, color, depth_scale=0.06, size_scale=0.11):
    side = h * size_scale; d = min(c, 32) * depth_scale
    for k in np.linspace(d, 0, max(2, min(c, 8))):
        ax.add_patch(Rectangle((x + k, k), side, side, fc=color, ec="white", lw=.8, alpha=.9))
    ax.text(x + (side + d) / 2, -0.5, label, ha="center", va="top", fontsize=9, color=NAVY)
    return x + side + d

fig, ax = plt.subplots(figsize=(15, 3.6)); ax.axis("off")
xpos = 0
specs = [(1, 28, "input\n1×28×28", BLUE), (16, 26, "conv 3×3 + ReLU\n16×26×26", ORANGE), (16, 13, "max-pool 2×2\n16×13×13", GREEN),
         (32, 11, "conv 3×3 + ReLU\n32×11×11", ORANGE), (32, 5, "max-pool 2×2\n32×5×5", GREEN)]
for c, h, lbl, col in specs:
    end = draw_volume(ax, xpos, c, h, lbl, col)
    ax.annotate("", xy=(end + .6, 1.2), xytext=(end + .1, 1.2), arrowprops=dict(arrowstyle="-|>", color=GREY))
    xpos = end + .8
for n, lbl in [(800, "flatten\n800"), (128, "dense + ReLU\n128"), (10, "dense\n10 classes")]:
    hgt = 0.6 + 2.6 * np.log(n) / np.log(800)
    ax.add_patch(Rectangle((xpos, 1.6 - hgt / 2), .35, hgt, fc=LAVENDER if n > 10 else ORANGE, ec=NAVY))
    ax.text(xpos + .17, -0.5, lbl, ha="center", va="top", fontsize=9, color=NAVY)
    if n != 10: ax.annotate("", xy=(xpos + 1.6, 1.6), xytext=(xpos + .45, 1.6), arrowprops=dict(arrowstyle="-|>", color=GREY))
    xpos += 1.9
ax.text(6.5, 4.4, "feature extraction", ha="center", fontsize=12, color=NAVY, fontweight="bold")
ax.text(xpos - 3.0, 4.4, "classifier", ha="center", fontsize=12, color=NAVY, fontweight="bold")
ax.set_xlim(-.3, xpos); ax.set_ylim(-1.6, 4.8); ax.set_aspect("equal"); plt.show()

---
# 6 · Why are CNNs better for vision?

**The problem with MLPs on images:** a 224 × 224 × 3 image has 150,528 inputs; a single dense layer with 1000 neurons needs
**150.5 M** weights. That is computationally explosive, prone to overfitting, and ignores spatial structure.

**The CNN solution: three key properties**
1. **Parameter sharing:** a 3 × 3 kernel has 9 weights, reused at every position of the image.
2. **Local connectivity:** each output only looks at a small neighbourhood (its *receptive field*).
3. **Translation equivariance:** a pattern is detected wherever it appears (Section 4); pooling adds some invariance.

In [ ]:
mlp_ref = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
cnn_ref = SimpleCNN()

def per_layer(model):
    return [(f"{type(m).__name__} {tuple(m.weight.shape)}", count_params(m)) for m in model.modules() if isinstance(m, (nn.Linear, nn.Conv2d))]

fig, axes = plt.subplots(1, 2, figsize=(14, 3.6), gridspec_kw=dict(width_ratios=[1, 1.6]))
axes[0].bar(["MLP\n(notebook 01)", "CNN\n(this notebook)"], [count_params(mlp_ref), count_params(cnn_ref)], color=[GREY, NAVY])
for i, v in enumerate([count_params(mlp_ref), count_params(cnn_ref)]): axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontweight="bold")
axes[0].set_title("Total parameters on MNIST"); axes[0].set_ylim(0, 270_000)
rows = [("MLP", n, v) for n, v in per_layer(mlp_ref)] + [("CNN", n, v) for n, v in per_layer(cnn_ref)]
axes[1].barh([f"{m}: {n}" for m, n, v in rows][::-1], [v for *_, v in rows][::-1],
             color=[GREY if m == "MLP" else (ORANGE if "Conv" in n else NAVY) for m, n, v in rows][::-1])
axes[1].set_xscale("log"); axes[1].set_xlabel("parameters (log scale)"); axes[1].set_title("Parameters per layer (conv layers in orange)")
plt.tight_layout(); plt.show()

big_fc, big_conv = 224 * 224 * 3 * 1000, 3 * 3 * 3 * 64 + 64
print(f"224×224×3 image → dense layer with 1000 neurons: {big_fc:>12,} weights")
print(f"224×224×3 image → conv layer with 64 3×3 kernels: {big_conv:>11,} weights  (independent of image size!)")

Almost all of our CNN's parameters sit in the **dense** layer after flattening. The two conv layers that do the actual
feature extraction have fewer than 5,000 parameters together.

---
# ✏️ Exercise 3: our first Convolutional Neural Network

**Tasks**
1. build a CNN with 2 conv layers (we did that above: `SimpleCNN`, take a look at it again!)
2. apply `MaxPool2d` after each conv
3. flatten and add 2 fully-connected layers
4. train with cross-entropy
5. plot training & test loss per epoch
6. compare the accuracy with Exercise 2 (the MLP)

🎯 **Goal:** > 98.5 % test accuracy with **fewer** parameters than the MLP.

We use the same data and the same `fit` helper as notebook 01.

In [ ]:
# @title Data & training helpers (same as notebook 01) { display-mode: "form" }
from torch.utils.data import TensorDataset, DataLoader
MEAN, STD = 0.1307, 0.3081
X_train = ((mnist_train.data.float() / 255) - MEAN) / STD
X_test = ((mnist_test.data.float() / 255) - MEAN) / STD
y_train, y_test = mnist_train.targets, mnist_test.targets

def make_loaders(X_tr, y_tr, X_te, y_te, batch_size=128):
    return (DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True),
            DataLoader(TensorDataset(X_te, y_te), batch_size=1000))

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device); logits = model(x)
        loss += F.cross_entropy(logits, y, reduction="sum").item(); correct += (logits.argmax(1) == y).sum().item(); n += len(y)
    return loss / n, correct / n

def fit(model, train_loader, test_loader, epochs=5, lr=1e-3):
    model.to(device); opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"train_loss": [], "test_loss": [], "test_acc": []}
    for epoch in range(epochs):
        model.train(); t0, running = time.time(), 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step(); running += loss.item() * len(y)
        tl, ta = evaluate(model, test_loader)
        hist["train_loss"].append(running / len(train_loader.dataset)); hist["test_loss"].append(tl); hist["test_acc"].append(ta)
        print(f"epoch {epoch + 1}/{epochs}  train loss {hist['train_loss'][-1]:.4f}  test loss {tl:.4f}  test acc {ta:.2%}  ({time.time() - t0:.1f}s)")
    return hist

train_loader, test_loader = make_loaders(X_train, y_train, X_test, y_test)

### ✏️ Build your own CNN
Write `MyCNN` yourself, *without* scrolling up. Hints: `nn.Conv2d(in_channels, out_channels, kernel_size)`, `nn.MaxPool2d(2)`, `nn.Flatten()`.
Use the output-size formula to work out the input size of the first `nn.Linear`!

In [ ]:
class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: self.features = nn.Sequential(conv → ReLU → pool → conv → ReLU → pool)
        # TODO: self.classifier = nn.Sequential(flatten → linear → ReLU → linear)
        pass

    def forward(self, x):
        x = x.unsqueeze(1)                    # (B, 28, 28) → (B, 1, 28, 28): one input channel
        return self.classifier(self.features(x))

In [ ]:
# @title 💡 Solution { display-mode: "form" }
MyCNN = SimpleCNN

In [ ]:
torch.manual_seed(0)
cnn = MyCNN()
print(f"CNN parameters: {count_params(cnn):,}")
hist_cnn = fit(cnn, train_loader, test_loader, epochs=5)

### Compare with the MLP
We retrain the MLP from notebook 01 with the identical setup, so the comparison is fair.

In [ ]:
torch.manual_seed(0)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
hist_mlp = fit(mlp, train_loader, test_loader, epochs=5)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
ep = range(1, 6)
for h, name, c in [(hist_mlp, "MLP", GREY), (hist_cnn, "CNN", NAVY)]:
    axes[0].plot(ep, h["train_loss"], "-o", color=c, label=f"{name} train"); axes[0].plot(ep, h["test_loss"], "--s", color=c, label=f"{name} test")
    axes[1].plot(ep, h["test_acc"], "-o", color=c, label=name)
axes[0].set(xlabel="epoch", ylabel="loss", title="Loss"); axes[0].legend(ncol=2, fontsize=8)
axes[1].set(xlabel="epoch", ylabel="test accuracy", title="Test accuracy"); axes[1].legend()
names = ["MLP", "CNN"]; params = [count_params(mlp), count_params(cnn)]; accs = [hist_mlp["test_acc"][-1], hist_cnn["test_acc"][-1]]
axes[2].axis("off")
table = axes[2].table(cellText=[[f"{p:,}" for p in params], [f"{a:.2%}" for a in accs], [f"{round((1 - a) * 10000)}" for a in accs]],
                      rowLabels=["parameters", "test accuracy", "errors / 10 000"], colLabels=names, loc="center", cellLoc="center")
table.scale(1, 2.2); table.set_fontsize(11); axes[2].set_title("Same data, fewer parameters, better results")
plt.tight_layout(); plt.show()

---
# 7 · What did the CNN learn?
The 16 kernels of the first layer, and what they produce for one test digit:

In [ ]:
cnn.eval()
w1 = cnn.features[0].weight.detach().cpu()[:, 0]          # (16, 3, 3)
x = X_test[:1].to(device)
with torch.no_grad():
    a1 = cnn.features[:2](x.unsqueeze(1))[0].cpu()          # after conv1 + ReLU: (16, 26, 26)
    a2 = cnn.features[:5](x.unsqueeze(1))[0].cpu()          # after conv2 + ReLU: (32, 11, 11)

fig = plt.figure(figsize=(16, 5.2))
gs = fig.add_gridspec(3, 19)
ax = fig.add_subplot(gs[:, 0:3]); ax.imshow(mnist_test.data[0], cmap="gray_r"); ax.set_title("input"); ax.axis("off")
rows = [(w1, "conv1 kernels (3×3)", "coolwarm"), (a1, "conv1 feature maps (26×26)", "magma"), (a2[:16], "conv2 feature maps (11×11, first 16 of 32)", "magma")]
for r, (maps, title, cmap) in enumerate(rows):
    for k in range(16):
        ax = fig.add_subplot(gs[r, 3 + k]); ax.imshow(maps[k], cmap=cmap); ax.axis("off")
        if k == 0: ax.set_title(title, loc="left", fontsize=10)
plt.show()

Several first-layer kernels look like little edge detectors (one side positive, the other negative), much like Sobel.
Their feature maps highlight strokes of a certain orientation. Nobody told the network to do this; it **learned** it.

---
# 8 · The shuffle experiment, revisited
In notebook 01 the MLP didn't care whether we shuffled the pixels. What about the CNN?

In [ ]:
torch.manual_seed(42)
perm = torch.randperm(784)
shuffle_pixels = lambda X: X.reshape(len(X), -1)[:, perm].reshape(-1, 28, 28)
loaders_perm = make_loaders(shuffle_pixels(X_train), y_train, shuffle_pixels(X_test), y_test)

torch.manual_seed(0); hist_cnn_perm = fit(SimpleCNN(), *loaders_perm, epochs=5)
torch.manual_seed(0); hist_mlp_perm = fit(nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10)), *loaders_perm, epochs=5)

fig, ax = plt.subplots(figsize=(7, 3.6))
vals = {"MLP": [hist_mlp["test_acc"][-1], hist_mlp_perm["test_acc"][-1]], "CNN": [hist_cnn["test_acc"][-1], hist_cnn_perm["test_acc"][-1]]}
xs = np.arange(2)
for k, (name, c) in enumerate([("MLP", GREY), ("CNN", NAVY)]):
    b = ax.bar(xs + (k - .5) * .38, vals[name], .36, color=c, label=name)
    ax.bar_label(b, labels=[f"{v:.1%}" for v in vals[name]], fontsize=9)
ax.set_xticks(xs, ["original images", "shuffled pixels"]); ax.set_ylim(0.9, 1.0); ax.set_ylabel("test accuracy"); ax.legend(loc="lower left")
ax.set_title("Who uses the spatial structure?"); plt.show()

On shuffled pixels the CNN **loses its advantage**: neighbouring pixels are no longer related, so small local kernels
have nothing to detect. On real images, with real spatial structure, it wins. The CNN's design (*locality* and *weight sharing*)
is an **inductive bias**: an assumption about the data that is built into the architecture. When the assumption holds it helps a lot.
Keep this word in mind for the Vision Transformer in notebook 04.

---
# 9 · What deep CNNs learn: a hierarchy of features
Our CNN has 2 conv layers. Real-world CNNs have dozens. Let's look inside **ResNet-18**, pretrained on 1.28 M ImageNet photos.
Early layers detect simple patterns; deeper layers respond to increasingly complex structures.

*Maritime:* edges → wave texture → hull shape → vessel type

In [ ]:
import torchvision
from torchvision.models import ResNet18_Weights
weights = ResNet18_Weights.DEFAULT
resnet = torchvision.models.resnet18(weights=weights).eval().to(device)
img = load_image("trawler", size=224)
x = weights.transforms()(img)[None].to(device)

acts = {}
hooks = [getattr(resnet, n).register_forward_hook(lambda m, i, o, n=n: acts.__setitem__(n, o[0].cpu())) for n in ["layer1", "layer2", "layer3", "layer4"]]
with torch.no_grad(): logits = resnet(x)
for h in hooks: h.remove()

# The 64 RGB kernels of the very first layer (7×7)
k = resnet.conv1.weight.detach().cpu()
k = (k - k.amin(dim=(1, 2, 3), keepdim=True)) / (k.amax(dim=(1, 2, 3), keepdim=True) - k.amin(dim=(1, 2, 3), keepdim=True))
fig, axes = plt.subplots(4, 16, figsize=(14, 3.8))
for i, ax in enumerate(axes.ravel()): ax.imshow(k[i].permute(1, 2, 0)); ax.axis("off")
plt.suptitle("ResNet-18, first layer: 64 learned 7×7 RGB kernels → edges, stripes and colour blobs", color=NAVY, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(4, 7, figsize=(15, 9))
info = {"layer1": "Layer 1 · 56×56\nedges & gradients", "layer2": "Layer 2 · 28×28\ncorners & textures",
        "layer3": "Layer 3 · 14×14\nobject parts", "layer4": "Layer 4 · 7×7\nwhole objects"}
for r, (name, a) in enumerate(acts.items()):
    axes[r, 0].imshow(img); axes[r, 0].set_ylabel(info[name], fontsize=11, color=NAVY, rotation=0, ha="right", va="center")
    top = a.mean((1, 2)).argsort(descending=True)[:6]               # the 6 most active channels
    for c, ch in enumerate(top):
        axes[r, c + 1].imshow(a[ch], cmap="magma"); axes[r, c + 1].set_title(f"channel {ch.item()}", fontsize=8)
for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Feature maps get coarser and more abstract with depth", color=NAVY, fontweight="bold"); plt.tight_layout(); plt.show()

print("ResNet-18 top-3 prediction:", [weights.meta["categories"][i] for i in logits.softmax(1)[0].topk(3).indices])

Notice how the maps shrink (56 → 7 pixels) while each value "sees" a larger part of the image: its **receptive field**
grows with every conv and pooling layer. Deep in the network a single value summarises a whole region of the boat.

---
# Summary
1. A **convolution** slides a small kernel over the image and computes a dot product at each position: a local pattern detector.
2. Output size: $\lfloor (W - K + 2p)/s \rfloor + 1$. **Stride** down-samples, **padding** preserves the border, **pooling** summarises.
3. A CNN stacks **Conv → ReLU → Pool** blocks for feature extraction, followed by a dense classifier.
4. CNNs beat MLPs on images thanks to **parameter sharing**, **local connectivity** and **translation equivariance**:
   fewer parameters (108 k vs. 235 k), better accuracy (~99 % vs. ~97.5 %).
5. These properties are an **inductive bias**: great when the data has spatial structure, useless when it does not (shuffled pixels).
6. Deep CNNs learn a **hierarchy**: edges → textures → parts → objects.

> **Take-home message:** CNNs are the backbone of modern computer vision. Understanding them is essential for any maritime vision application.

**Next session →** [03 · Vision Foundation Models](https://colab.research.google.com/github/TorbenGl/visionseminar/blob/main/notebooks/03_vision_foundation_models.ipynb):
networks pretrained once on millions of images and reused everywhere.